In [1]:
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input


In [2]:
DATASET_PATH = "dataset_final_pp_cropped"
IMG_SIZE = 300
BATCH_SIZE = 16
EPOCHS_STAGE1 = 30
EPOCHS_STAGE2 = 30

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2), # Increased rotation
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1), # Added translation
    tf.keras.layers.RandomZoom(0.2), # Increased zoom
    tf.keras.layers.RandomContrast(0.2), # Increased contrast
])

In [3]:
# =========================
# BUILD LABEL MAPS
# =========================

species_map = {"cow": 0, "buffalo": 1}
group_map = {}
breed_map = {}

image_paths = []
species_labels = []
group_labels = []
breed_labels = []

group_index = 0
breed_index = 0

for species in os.listdir(DATASET_PATH):
    if species.startswith("."):
        continue

    species_path = os.path.join(DATASET_PATH, species)

    if species == "cow":
        for group in os.listdir(species_path):
            if group.startswith("."):
                continue

            group_path = os.path.join(species_path, group)

            if group not in group_map:
                group_map[group] = group_index
                group_index += 1

            for breed in os.listdir(group_path):
                if breed.startswith("."):
                    continue

                breed_path = os.path.join(group_path, breed)

                if breed not in breed_map:
                    breed_map[breed] = breed_index
                    breed_index += 1

                for file in os.listdir(breed_path):
                    if file.lower().endswith((".jpg", ".jpeg", ".png")):
                        image_paths.append(os.path.join(breed_path, file))
                        species_labels.append(species_map["cow"])
                        group_labels.append(group_map[group])
                        breed_labels.append(breed_map[breed])

    elif species == "buffalo":
        if "buffalo" not in group_map:
            group_map["buffalo"] = group_index
            group_index += 1

        for breed in os.listdir(species_path):
            if breed.startswith("."):
                continue

            breed_path = os.path.join(species_path, breed)

            if breed not in breed_map:
                breed_map[breed] = breed_index
                breed_index += 1

            for file in os.listdir(breed_path):
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    image_paths.append(os.path.join(breed_path, file))
                    species_labels.append(species_map["buffalo"])
                    group_labels.append(group_map["buffalo"])
                    breed_labels.append(breed_map[breed])





In [4]:
image_paths = np.array(image_paths)
species_labels = np.array(species_labels)
group_labels = np.array(group_labels)
breed_labels = np.array(breed_labels)

NUM_GROUPS = len(group_map)
NUM_BREEDS = len(breed_map)
breed_names = list(breed_map.keys())

In [5]:
# =========================
# CLASS WEIGHTS
# =========================
# Calculate class weights for 'breed' since dataset is imbalanced
unique_breeds = np.unique(breed_labels)
class_weights_vals = compute_class_weight(class_weight='balanced', classes=unique_breeds, y=breed_labels)
breed_weight_dict = dict(zip(unique_breeds, class_weights_vals))

# Map weights to each sample
sample_weights = np.array([breed_weight_dict[b] for b in breed_labels])

In [6]:
# =========================
# TRAIN/VAL SPLIT
# =========================

(train_paths, val_paths,
 train_species, val_species,
 train_group, val_group,
 train_breed, val_breed,
 train_weights, val_weights) = train_test_split(
    image_paths,
    species_labels,
    group_labels,
    breed_labels,
    sample_weights,
    test_size=0.2,
    random_state=42,
    stratify=breed_labels
)

In [7]:
# =========================
# DATA PIPELINE
# =========================

def load_image(path, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

    if augment:
        img = data_augmentation(img)

    img = preprocess_input(img)
    return img

def create_dataset(paths, species, group, breed, weights, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, species, group, breed, weights))

    def process(path, sp, gr, br, wt):
        img = load_image(path, augment)
        targets = {
            "species": sp,
            "group": gr,
            "breed": br
        }
        weights_dict = {
            "species": tf.constant(1.0, dtype=tf.float32),
            "group": tf.constant(1.0, dtype=tf.float32),
            "breed": tf.cast(wt, tf.float32)
        }
        return img, targets, weights_dict

    ds = ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_species, train_group, train_breed, train_weights, augment=True)
val_ds = create_dataset(val_paths, val_species, val_group, val_breed, val_weights, augment=False)


In [8]:
# =========================
# MODEL
# =========================

# Upgraded to EfficientNetB3 which natively expects 300x300 images
base_model = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_breed_accuracy",
    patience=6,
    restore_best_weights=True,
    mode="max"
)
base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

species_out = layers.Dense(2, activation="softmax", name="species")(x)
group_out = layers.Dense(NUM_GROUPS, activation="softmax", name="group")(x)
breed_out = layers.Dense(NUM_BREEDS, activation="softmax", name="breed")(x)

# Changed from list to dictionary here
model = Model(inputs=inputs, outputs={
    "species": species_out,
    "group": group_out,
    "breed": breed_out
})

# Increased initial learning rate (1e-3 instead of 1e-5)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "species": "sparse_categorical_crossentropy",
        "group": "sparse_categorical_crossentropy",
        "breed": "sparse_categorical_crossentropy"
    },
    loss_weights={
        "species": 0.3,
        "group": 0.7,
        "breed": 3.0
    },
    metrics={
        "species": "accuracy",
        "group": "accuracy",
        "breed": "accuracy"
    }
)


In [9]:
# =========================
# STAGE 1 TRAINING
# =========================

print("\\nStarting Stage 1 Training (Base Model Frozen)...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=[early_stop]
)

\nStarting Stage 1 Training (Base Model Frozen)...
Epoch 1/30
223/223 ━━━━━━━━━━━━━━━━━━━━ 244s 1s/step - breed_accuracy: 0.2096 - breed_loss: 2.9388 - group_accuracy: 0.5865 - group_loss: 1.1654 - loss: 9.7115 - species_accuracy: 0.9038 - species_loss: 0.2640 - val_breed_accuracy: 0.4297 - val_breed_loss: 2.1106 - val_group_accuracy: 0.8459 - val_group_loss: 0.6385 - val_loss: 6.8095 - val_species_accuracy: 0.9741 - val_species_loss: 0.1045
Epoch 2/30
223/223 ━━━━━━━━━━━━━━━━━━━━ 211s 948ms/step - breed_accuracy: 0.4675 - breed_loss: 1.8609 - group_accuracy: 0.7961 - group_loss: 0.6189 - loss: 6.0503 - species_accuracy: 0.9516 - species_loss: 0.1143 - val_breed_accuracy: 0.5433 - val_breed_loss: 1.7901 - val_group_accuracy: 0.8616 - val_group_loss: 0.5346 - val_loss: 5.7639 - val_species_accuracy: 0.9741 - val_species_loss: 0.0897
Epoch 3/30
223/223 ━━━━━━━━━━━━━━━━━━━━ 209s 938ms/step - breed_accuracy: 0.5470 - breed_loss: 1.6314 - group_accuracy: 0.8053 - group_loss: 0.5733 - loss: 

In [ ]:
# =========================
# STAGE 2 FINE-TUNING
# =========================

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Scaled down learning rate for fine tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        "species": "sparse_categorical_crossentropy",
        "group": "sparse_categorical_crossentropy",
        "breed": "sparse_categorical_crossentropy"
    },
    loss_weights={
        "species": 0.3,
        "group": 0.7,
        "breed": 3.0
    },
    metrics={
        "species": "accuracy",
        "group": "accuracy",
        "breed": "accuracy"
    }
)

print("\\nStarting Stage 2 Fine-Tuning...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=[early_stop]
)

\nStarting Stage 2 Fine-Tuning...
Epoch 1/30


In [ ]:
# =========================
# PLOTS
# =========================

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history2.history["breed_accuracy"], label="Train Breed Acc")
plt.plot(history2.history["val_breed_accuracy"], label="Val Breed Acc")
plt.legend()
plt.title("Breed Accuracy")

plt.subplot(1,2,2)
plt.plot(history2.history["group_accuracy"], label="Train Group Acc")
plt.plot(history2.history["val_group_accuracy"], label="Val Group Acc")
plt.legend()
plt.title("Group Accuracy")

plt.show()

In [ ]:
# =========================
# CONFUSION MATRIX
# =========================

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    breed_preds = np.argmax(preds["breed"], axis=1)

    y_pred.extend(breed_preds)
    y_true.extend(labels.numpy())

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_BREEDS))
)

plt.figure(figsize=(14,14))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=breed_names
)

disp.plot(xticks_rotation=90)
plt.title("Breed Confusion Matrix")
plt.show()


In [ ]:
# =========================
# SAVE MODEL
# =========================

os.makedirs("backend_model", exist_ok=True)
model.save("backend_model/new_cattle_classifier.keras")

with open("backend_model/breed_names.json", "w", encoding="utf-8") as f:
    json.dump(breed_names, f, indent=4)

print("\\nSuccessfully saved trained model to 'backend_model/new_cattle_classifier.keras' and breed maps to 'backend_model/breed_names.json'.")
